# 03 – Data enrichment (Wikipedia)

For the feature films released between 1980 and 2025, the project adds a
synopsis to the top 10 films of each genre and year, and a short biography
to their first three principals. This notebook explains each step of the
pipeline and shows the result.

**Prerequisites** (after the data preparation, inside the Spark container):

```bash
docker compose exec spark python jobs/enrichment.py targets
docker compose exec spark python scripts/fetch_wikipedia.py      # a few hours, resumable
docker compose exec spark python jobs/enrichment.py build
```

Code: `src/imdb_project/enrichment.py` (Spark processing),
`scripts/fetch_wikipedia.py` (download), `jobs/enrichment.py` (steps).

In [1]:
import pandas as pd
from pyspark.sql import functions as F

from imdb_project.spark import get_spark

spark = get_spark("notebook-enrichment")
spark.sparkContext.setLogLevel("ERROR")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

from imdb_project import enrichment as en
from imdb_project.config import ENRICHMENT_DIR

targets = ENRICHMENT_DIR / "targets"
films = spark.read.parquet(str(targets / "films"))
people = spark.read.parquet(str(targets / "people"))
enriched = spark.read.parquet(str(ENRICHMENT_DIR / "enriched_films" / "parquet"))

## 1. Selecting the films and the principals (Spark)

- **Films**: feature films (`movie`, over 40 minutes) released between 1980
  and 2025, with at least 1,000 votes. For each (genre, year), the 10
  best-rated films are kept (ties broken by the number of votes). A film can
  be in the top 10 of several genres; it is enriched once. A (genre, year)
  with fewer than 10 films of 1,000 votes or more gives fewer films.
- **Principals**: the first three people of each film in the IMDb credit
  order (`title.principals.ordering`), whatever their category: this is the
  order of importance given by IMDb (lead actors, sometimes the director).

**Why the rating.** The question asks for the "top 10 films": ranking by
rating among reliably rated films (at least 1,000 votes, `MIN_VOTES` in
`definitions.py`) selects the best films of each genre and year; ranking by
number of votes would select the most watched ones.

In [2]:
print(f"{films.count():,} films, {people.count():,} (film, principal) pairs, "
      f"{people.select('nconst').distinct().count():,} distinct people")
films.orderBy(F.desc("numVotes")).limit(10).toPandas()

6,258 films, 18,773 (film, principal) pairs, 11,366 distinct people


,tconst,primaryTitle,startYear,averageRating,numVotes,top_in
0,tt0111161,The Shawshank Redemption,1994,9.3,3263256,"[{'genre': 'Drama', 'rank': 1}]"
1,tt0468569,The Dark Knight,2008,9.1,3249082,"[{'genre': 'Crime', 'rank': 1}, {'genre': 'Thriller', 'rank': 1}]"
2,tt1375666,Inception,2010,8.8,2888117,"[{'genre': 'Adventure', 'rank': 1}, {'genre': 'Sci-Fi', 'rank': 1}, {'genre': 'Thrille..."
3,tt0137523,Fight Club,1999,8.8,2682182,"[{'genre': 'Crime', 'rank': 1}, {'genre': 'Drama', 'rank': 1}, {'genre': 'Thriller', '..."
4,tt0816692,Interstellar,2014,8.7,2632258,"[{'genre': 'Adventure', 'rank': 1}, {'genre': 'Sci-Fi', 'rank': 1}]"
5,tt0109830,Forrest Gump,1994,8.8,2552214,"[{'genre': 'Drama', 'rank': 3}, {'genre': 'Romance', 'rank': 1}]"
6,tt0110912,Pulp Fiction,1994,8.8,2482559,"[{'genre': 'Crime', 'rank': 1}, {'genre': 'Drama', 'rank': 4}]"
7,tt0133093,The Matrix,1999,8.7,2293136,"[{'genre': 'Action', 'rank': 1}, {'genre': 'Sci-Fi', 'rank': 1}]"
8,tt0120737,The Lord of the Rings: The Fellowship of the Ring,2001,8.9,2251341,"[{'genre': 'Adventure', 'rank': 1}, {'genre': 'Drama', 'rank': 1}, {'genre': 'Fantasy'..."
9,tt0167260,The Lord of the Rings: The Return of the King,2003,9.0,2207007,"[{'genre': 'Adventure', 'rank': 1}, {'genre': 'Drama', 'rank': 1}, {'genre': 'Fantasy'..."


In [3]:
# Number of films per (genre, year): fewer than 10 when not enough films have 1,000 votes.
per_genre_year = films.select(F.explode("top_in").alias("t")).groupBy("t.genre").agg(
    F.count(F.lit(1)).alias("films"))
per_genre_year.orderBy("films").toPandas()

,genre,films
0,Talk-Show,1
1,Adult,6
2,News,28
3,Western,152
4,Musical,374
5,Sport,397
6,Documentary,399
7,War,421
8,History,444
9,Animation,445


## 2. Downloading the articles (outside Spark)

The instructions allow the data gathering to be done outside Spark.
`scripts/fetch_wikipedia.py`:

1. maps each IMDb id to its English Wikipedia article through **Wikidata**,
   which stores the IMDb id of films and people (property P345). There is
   no search by title, so no wrong article can be picked for a homonym;
2. downloads the plain text of each film article and the lead section
   (introduction) of each person's article with the MediaWiki API.

**Politeness and provenance.**

- one request at a time, one per second, with `maxlag` on the MediaWiki
  API (the server can ask the client to wait when it is busy);
- a User-Agent naming the project and its repository, as asked by the
  Wikimedia User-Agent policy;
- retries with a growing pause on HTTP 429 / 5xx and network errors,
  honouring `Retry-After`;
- each record keeps the Wikipedia URL, the Wikidata URL and the download
  time; the script can be stopped and restarted where it stopped.

**Sources.** Wikipedia and Wikidata are written by people (no machine
generated text) and published under open licences (CC BY-SA and CC0).

In [4]:
raw_films = en.read_raw(spark, ENRICHMENT_DIR / "raw" / "films.jsonl")
raw_people = en.read_raw(spark, ENRICHMENT_DIR / "raw" / "people.jsonl")
for name, raw in [("films", raw_films), ("people", raw_people)]:
    print(f"--- {name}")
    print(raw.groupBy("status").count().toPandas().to_string(index=False))

--- films


    status  count
        ok   8960
no_article   1183
--- people


    status  count
        ok  10938
no_article   5227


In [5]:
raw_films.select("imdb_id", "wikipedia_title", "source_url", "wikidata_url", "retrieved_at",
                 "status").limit(5).toPandas()

,imdb_id,wikipedia_title,source_url,wikidata_url,retrieved_at,status
0,tt0035423,Kate & Leopold,https://en.wikipedia.org/wiki/Kate_%26_Leopold,https://www.wikidata.org/wiki/Q1334550,2026-11-10T19:52:32+00:00,ok
1,tt0061876,Commissar (film),https://en.wikipedia.org/wiki/Commissar_%28film%29,https://www.wikidata.org/wiki/Q1213999,2026-11-10T19:52:33+00:00,ok
2,tt0067625,Trial on the Road,https://en.wikipedia.org/wiki/Trial_on_the_Road,https://www.wikidata.org/wiki/Q3213961,2026-11-10T19:52:34+00:00,ok
3,tt0075788,None,None,https://www.wikidata.org/wiki/Q3645481,2026-11-10T19:52:31+00:00,no_article
4,tt0076276,Who's Singin' Over There?,https://en.wikipedia.org/wiki/Who%27s_Singin%27_Over_There%3F,https://www.wikidata.org/wiki/Q280962,2026-11-11T15:23:53+00:00,ok


## 3. Cleaning and assembling (Spark)

All the processing after the download is done with Spark:

- the raw JSON is read with an explicit schema (`RAW_SCHEMA`);
- **synopsis**: the section of the film article whose title is one of the
  usual names of the plot section is extracted; other articles get a note;
- **biography**: the lead section of the person's article;
- texts are cleaned: reference marks such as `[1]`, leftover math markup
  and repeated spaces or blank lines are removed;
- one row per film, with its synopsis, its principals (array of
  structures with their biography) and the source URL of every text;
- when a text is missing, a note explains why (`synopsis_note`,
  `biography_note`).

The functions take DataFrames and contain no limit on the number of films:
the same code runs on the full IMDb dataset, only the target selection
restricts the scope.

In [6]:
print("Synopsis section names:", en.SYNOPSIS_SECTIONS)
pd.DataFrame(sorted(en.NOTES.items()), columns=["reason", "note"])

Synopsis section names: ['Plot', 'Plot summary', 'Synopsis', 'Premise', 'Story', 'Summary', 'Storyline']


,reason,note
0,empty_text,The Wikipedia article has no text for this field.
1,missing_page,The Wikipedia article linked in Wikidata could not be retrieved.
2,no_article,No English Wikipedia article is linked to this IMDb id in Wikidata.
3,no_synopsis_section,The Wikipedia article has no plot or synopsis section.
4,not_fetched,Not fetched yet: run scripts/fetch_wikipedia.py.


In [7]:
enriched.printSchema()

root
 |-- tconst: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- startYear: integer (nullable = true)
 |-- numVotes: integer (nullable = true)
 |-- top_in: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- genre: string (nullable = true)
 |    |    |-- rank: integer (nullable = true)
 |-- synopsis: string (nullable = true)
 |-- synopsis_words: integer (nullable = true)
 |-- synopsis_note: string (nullable = true)
 |-- synopsis_source_url: string (nullable = true)
 |-- wikidata_url: string (nullable = true)
 |-- principals: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- position: integer (nullable = true)
 |    |    |-- nconst: string (nullable = true)
 |    |    |-- primaryName: string (nullable = true)
 |    |    |-- category: string (nullable = true)
 |    |    |-- biography: string (nullable = true)
 |    |    |-- biography_note: string (nullable = true)
 |    |    |-- biography_s

## 4. Result

In [8]:
pd.read_csv(next((ENRICHMENT_DIR / "coverage").glob("*.csv")))

,item,targets,enriched,enriched_pct
0,films,6258,5320,85.0
1,film principals,18773,15370,81.9


In [9]:
enriched.groupBy("synopsis_note").count().orderBy(F.desc("count")).toPandas()

,synopsis_note,count
0,None,5320
1,The Wikipedia article has no plot or synopsis section.,497
2,No English Wikipedia article is linked to this IMDb id in Wikidata.,441


In [10]:
principals = enriched.select(F.explode("principals").alias("p")).select("p.*")
principals.groupBy("biography_note").count().orderBy(F.desc("count")).toPandas()

,biography_note,count
0,None,15370
1,No English Wikipedia article is linked to this IMDb id in Wikidata.,3402
2,The Wikipedia article has no text for this field.,1


In [11]:
# One enriched film.
film = enriched.where("tconst = 'tt1375666'").first() or enriched.first()
print(film["primaryTitle"], film["startYear"], "-", film["synopsis_source_url"])
print(film["synopsis"][:800], "...\n")
for p in film["principals"]:
    print(f"{p['position']}. {p['primaryName']} ({p['category']}) - {p['biography_source_url']}")
    print("  ", (p["biography"] or p["biography_note"])[:300], "...")

Inception 2010 - https://en.wikipedia.org/wiki/Inception
Dom Cobb and Arthur are professional thieves who perform corporate espionage using experimental dream-sharing technology to infiltrate their targets' subconscious and extract information. Their latest target Saito is impressed with Cobb's ability to layer multiple dreams within each other. He offers to hire Cobb for the ostensibly impossible job of performing "inception": implanting Robert Fischer, the son of Saito's competitor Maurice Fischer, with the idea to dissolve his father's conglomerate. In return, Saito promises to clear Cobb's criminal status, allowing him to return home to his children.
Cobb accepts the offer and recruits a forger named Eames, a chemist named Yusuf, and an architecture student named Ariadne. They devise a three-level dream structure in order for Robert to accep ...

1. Leonardo DiCaprio (actor) - https://en.wikipedia.org/wiki/Leonardo_DiCaprio
   Leonardo Wilhelm DiCaprio ( ; Italian: [diˈkaːprjo]; bo

In [12]:
spark.stop()